# HDA Digital Twin collections census



**Goal**: establish, collection by collection, which Digital Twin
collections in the HDA production catalog actually expose zarr assets
through the desp_cache route, and cross the result against the datasets
described on the
[Earth Data Hub climate dt 2 page](https://earthdatahub.destine.eu/collections/climate-dt-2).

For each DT collection the census records: the HTTP status of a plain items
query, the number of items returned, whether any item carries a `.zarr`
asset, whether that asset href goes through `desp_cache`, and the first line
of the error body when the query fails (a 400 here usually means the
collection is order based and expects search constraints instead of a plain
listing).

In [1]:
from getpass import getpass

import destinelab as deauth
import pandas as pd
import requests

HDA_STAC_API = "https://hda.data.destination-earth.eu/stac/v2"

DESP_USERNAME = input("DESP username: ")
DESP_PASSWORD = getpass("DESP password: ")
auth = deauth.AuthHandler(DESP_USERNAME, DESP_PASSWORD)
access_token = auth.get_token()
assert access_token is not None, "Failed to obtain access token"
headers = {"Authorization": f"Bearer {access_token}"}

In [2]:
# fetch all collections; the ".DT_" filter avoids the MDT_ false positive
resp = requests.get(f"{HDA_STAC_API}/collections", headers=headers)
resp.raise_for_status()
dt_ids = [c["id"] for c in resp.json()["collections"] if ".DT_" in c["id"]]
print(f"{len(dt_ids)} DT collections to probe\n")

rows = []
for cid in dt_ids:
    r = requests.get(f"{HDA_STAC_API}/collections/{cid}/items", headers=headers)
    row = {"collection": cid, "items_http": r.status_code, "n_items": None,
           "has_zarr": False, "via_desp_cache": False, "error": ""}
    if r.ok:
        feats = r.json().get("features", [])
        row["n_items"] = len(feats)
        for f in feats:
            for key, asset in f.get("assets", {}).items():
                if key.endswith(".zarr"):
                    row["has_zarr"] = True
                    if "desp_cache" in asset.get("href", ""):
                        row["via_desp_cache"] = True
    else:
        row["error"] = r.text[:120].replace("\n", " ")
    rows.append(row)
    print(f"{row['items_http']}  {cid}")

census = pd.DataFrame(rows)
census

40 DT collections to probe

400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_ICON.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_IFS-NEMO.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-NEMO.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_ICON.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_IFS-NEMO.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_CONT_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_HIST_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_TPLUS2.0K_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.BASELINE_CONT_ICON.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.BASELINE_CONT_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.BASELINE_CONT_IFS-NEMO.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.BASELINE_HIST_ICON.R1
400  EO.ECMWF.DAT.D1.DT_CLIMATE.G2.BASELINE_HIST_IFS-FESOM.R1
400  EO.ECMWF.DAT.D1.DT_CLI

,collection,items_http,n_items,has_zarr,via_desp_cache,error
0,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_ICON.R1,400,NaN,False,False,"{""code"":""400"",""ticket"":""1717facb"",""description..."
1,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_IFS-N...,400,NaN,False,False,"{""code"":""400"",""ticket"":""912e16d8"",""description..."
2,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_...,400,NaN,False,False,"{""code"":""400"",""ticket"":""5e94f642"",""description..."
3,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_...,400,NaN,False,False,"{""code"":""400"",""ticket"":""a9bd3dc4"",""description..."
4,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3...,400,NaN,False,False,"{""code"":""400"",""ticket"":""a605a289"",""description..."
5,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3...,400,NaN,False,False,"{""code"":""400"",""ticket"":""ba84006b"",""description..."
6,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3...,400,NaN,False,False,"{""code"":""400"",""ticket"":""42b6c72c"",""description..."
7,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_CO...,400,NaN,False,False,"{""code"":""400"",""ticket"":""e3c06501"",""description..."
8,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_HI...,400,NaN,False,False,"{""code"":""400"",""ticket"":""a39ee392"",""description..."
9,EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_TP...,400,NaN,False,False,"{""code"":""400"",""ticket"":""5556080e"",""description..."


In [3]:
# condensed view: which collections actually serve zarr through desp_cache
print("Collections WITH zarr via desp_cache:")
print(census[census.via_desp_cache]["collection"].to_string(index=False))
print()
print("Collections answering items but WITHOUT zarr assets:")
print(census[census.items_http.eq(200) & ~census.has_zarr]["collection"].to_string(index=False))
print()
print("Collections rejecting the plain items query:")
print(census[census.items_http.ne(200)][["collection", "items_http"]].to_string(index=False))

Collections WITH zarr via desp_cache:
    EO.ECMWF.DAT.DT_CLIMATE_ADAPTATION_ICON
EO.ECMWF.DAT.DT_CLIMATE_ADAPTATION_IFS-NEMO

Collections answering items but WITHOUT zarr assets:
Series([], )

Collections rejecting the plain items query:
                                                        collection  items_http
                  EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_ICON.R1         400
              EO.ECMWF.DAT.D1.DT_CLIMATE.G1.CMIP6_HIST_IFS-NEMO.R1         400
        EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-FESOM.R1         400
         EO.ECMWF.DAT.D1.DT_CLIMATE.G1.HIGHRESMIP_CONT_IFS-NEMO.R1         400
        EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_ICON.R1         400
   EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_IFS-FESOM.R1         400
    EO.ECMWF.DAT.D1.DT_CLIMATE.G1.SCENARIOMIP_SSP3-7.0_IFS-NEMO.R1         400
     EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_CONT_IFS-FESOM.R1         400
     EO.ECMWF.DAT.D1.DT_CLIMATE.G1.STORY-NUDGING_H

In [4]:
for cid in ["EO.ECMWF.DAT.DT_CLIMATE_ADAPTATION_IFS-NEMO",
            "EO.ECMWF.DAT.DT_CLIMATE_ADAPTATION_ICON"]:
    r = requests.get(f"{HDA_STAC_API}/collections/{cid}/items", headers=headers)
    r.raise_for_status()
    for f in r.json()["features"]:
        for k, a in f["assets"].items():
            if k.endswith(".zarr"):
                print(a["href"].split("/")[-1])


ScenarioMIP-SSP3-7.0-IFS-NEMO-0001-high-o2d-v0.zarr
ScenarioMIP-SSP3-7.0-IFS-NEMO-0001-high-sfc-monthly-v0.zarr
ScenarioMIP-SSP3-7.0-IFS-NEMO-0001-high-sfc-v0.zarr
ScenarioMIP-SSP3-7.0-IFS-NEMO-0001-high-pl-v0.zarr
ScenarioMIP-SSP3-7.0-ICON-0001-high-sfc-monthly-v0.zarr
ScenarioMIP-SSP3-7.0-ICON-0001-high-sfc-v0.zarr


In [5]:
# zarr format check for the 6 exposed stores:
# a zarr v2 store exposes a .zgroup file at its root, a zarr v3 store a zarr.json file
for cid in ["EO.ECMWF.DAT.DT_CLIMATE_ADAPTATION_IFS-NEMO",
            "EO.ECMWF.DAT.DT_CLIMATE_ADAPTATION_ICON"]:
    r = requests.get(f"{HDA_STAC_API}/collections/{cid}/items", headers=headers)
    r.raise_for_status()
    for f in r.json()["features"]:
        for k, a in f["assets"].items():
            if k.endswith(".zarr"):
                href = a["href"]
                v2 = requests.get(f"{href}/.zgroup", headers=headers).status_code
                v3 = requests.get(f"{href}/zarr.json", headers=headers).status_code
                fmt = "zarr v2" if v2 == 200 else ("zarr v3" if v3 == 200 else "unknown")
                print(f"{fmt}  {href.split('/')[-1]}")

zarr v2  ScenarioMIP-SSP3-7.0-IFS-NEMO-0001-high-o2d-v0.zarr
zarr v2  ScenarioMIP-SSP3-7.0-IFS-NEMO-0001-high-sfc-monthly-v0.zarr
zarr v2  ScenarioMIP-SSP3-7.0-IFS-NEMO-0001-high-sfc-v0.zarr
zarr v2  ScenarioMIP-SSP3-7.0-IFS-NEMO-0001-high-pl-v0.zarr
zarr v2  ScenarioMIP-SSP3-7.0-ICON-0001-high-sfc-monthly-v0.zarr
zarr v2  ScenarioMIP-SSP3-7.0-ICON-0001-high-sfc-v0.zarr


## Findings 
1. The split is exactly binary: of the 40 DT collections, **2 expose zarr
   through desp_cache** (`EO.ECMWF.DAT.DT_CLIMATE_ADAPTATION_IFS-NEMO`, 4
   items, and `EO.ECMWF.DAT.DT_CLIMATE_ADAPTATION_ICON`, 2 items) and **38
   are order based** (they reject an unconstrained items listing with HTTP
   400, which is their expected behavior: their items are generated from
   search constraints).
2. The 6 zarr stores served by those 2 collections belong to the **previous
   Climate DT release**, not to generation 2.
   * Shown by this notebook: the format check cell above reports the stores
     as **zarr v2**, while the generation 2 mirror is published in zarr v3
     (stated on the
     [climate dt 2 page](https://earthdatahub.destine.eu/collections/climate-dt-2)).
   * External references: the store names listed above match the datasets of
     the earlier
     [climate dt collection](https://earthdatahub.destine.eu/collections/climate-dt)
     and do not appear on the climate dt 2 page; the HDA team confirmed that
     the two collections predate generation 2.
3. Therefore, **none of the 36 generation 2 datasets** published in the
   cache (3 models, 3 scenarios, 4 datasets each) is registered in HDA
   today.
